<a href="https://colab.research.google.com/github/Dorkom/Automatizaciones_google_colab/blob/main/PRODUCCION_pipeline_final_ulima.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🧪 Pipeline final de extracción — Ulima → RAG

Extrae el contenido de la página institucional y lo deja **limpio y completo**
para alimentar una base vectorial.

**Diagnóstico (probado sobre el DOM renderizado de la página):**
- Todo el contenido vive en el HTML servido por el servidor (Drupal SSR). Los
  tabs son **sliders Slick + acordeones** ocultos por CSS, no contenido que se
  cargue al hacer click.
- Por eso basta `requests` (sin navegador): `BeautifulSoup.get_text()` ignora el
  CSS y captura también los paneles ocultos.
- El reto real no era *capturar*, sino **quitar el ruido/duplicación** del widget
  "Copiar enlace" de la plantilla, que repite cada encabezado 3–4 veces.

**Flujo:** `scrape_ulima()` → `clean_ulima_html()` (quita duplicados y ruido) →
`polish_text()` (quita chrome de slider) → QA de cobertura → guardado a disco.

> Ejecuta las celdas en orden. Solo la primera (setup) hace falta una vez por sesión.

## 1. Setup

Dependencias mínimas: no se necesita navegador porque la página es SSR.

In [ ]:
#!pip install -q "requests>=2.31" "beautifulsoup4>=4.12" "lxml>=5.0"
#print("✅ Setup listo.")

## 2. Configuración y tipo de resultado

`ScrapeResult` estandariza la salida (texto, tiempo, error, conteo de caracteres).

In [ ]:
from __future__ import annotations

import re
import time
from dataclasses import dataclass

import requests
from bs4 import BeautifulSoup

# --- Configuración ---------------------------------------------------------
TARGET_URL: str = "https://peru.workuse.com/work-and-travel-usa/requisitos/"
OUTPUT_PATH = "workuse - work-and-travel.txt"
HEADERS: dict[str, str] = {
    "User-Agent": "Mozilla/5.0 (compatible; ULimaRAGBot/1.0)"
}
REQUEST_TIMEOUT_S: int = 30


@dataclass
class ScrapeResult:
    """Resultado estandarizado de una extracción."""

    method: str
    text: str | None = None
    error: str | None = None
    elapsed_s: float = 0.0

    @property
    def ok(self) -> bool:
        return self.error is None and self.text is not None

    @property
    def char_count(self) -> int:
        return len(self.text) if self.text else 0


print(f"Configuración lista. URL:\n  {TARGET_URL}")

Configuración lista. URL:
  https://peru.workuse.com/work-and-travel-usa/requisitos/


## 3. Limpieza quirúrgica del HTML

Elimina, a nivel de DOM, las etiquetas estructurales y los selectores de ruido
específicos de la plantilla Drupal (los que duplican títulos y los menús
"Copiar enlace"). Captura el 100% del contenido sin la duplicación.

In [ ]:
from urllib.parse import urljoin   # nuevo import

# Etiquetas estructurales sin contenido útil.
_NOISE_TAGS: tuple[str, ...] = (
    "script", "style", "noscript", "svg", "iframe", "form",
    "header", "footer", "nav",
)

# Selectores de ruido específicos de la plantilla (identificados en el DOM):
#   - títulos de ancla ocultos / etiquetas de acordeón -> duplican cada heading
#   - menús contextuales "Copiar enlace"
_JUNK_SELECTORS: tuple[str, ...] = (
    ".jfu--component-anchor-title",
    ".jfu--component-anchor",
    ".block-title--accordion",
    ".visually-hidden",
    ".hidden",
    ".u-context-menu",
    ".ut-context-menu",
    "[class*='anchor']",
)

# Extensiones que consideramos "documentos descargables" (vitales para el RAG).
_DOC_EXTS: tuple[str, ...] = (
    ".pdf", ".doc", ".docx", ".xls", ".xlsx", ".ppt", ".pptx", ".csv",
)

def _is_doc_link(url: str) -> bool:
    clean = url.lower().split("?")[0].split("#")[0]
    return clean.endswith(_DOC_EXTS)

def clean_ulima_html(html: str, base_url: str = TARGET_URL) -> str:
    """Convierte el HTML de Ulima en texto limpio, completo y sin duplicados.

    Conserva las URLs de documentos descargables (PDF, Word, etc.), que
    `get_text()` descartaría por defecto.
    """
    soup = BeautifulSoup(html, "lxml")

    # 1) Fuera etiquetas estructurales (incluye nav/header/footer => quita
    #    enlaces de navegación, que NO queremos).
    for tag in soup.find_all(_NOISE_TAGS):
        tag.decompose()

    # 2) Captura enlaces a documentos ANTES de la limpieza de ruido.
    #    Los inserta inline para que sobrevivan a get_text() y mantengan
    #    su contexto (en qué sección aparecen).
    doc_links: list[tuple[str, str]] = []   # (texto_visible, url_absoluta)
    seen: set[str] = set()
    for a in soup.find_all("a", href=True):
        href = a["href"].strip()
        if not href or href.startswith(("#", "javascript:", "mailto:", "tel:")):
            continue
        url = urljoin(base_url, href)        # /sites/... -> URL absoluta
        if not _is_doc_link(url):
            continue
        label = a.get_text(" ", strip=True) or url
        a.replace_with(f"{label} ({url})")
        if url not in seen:
            seen.add(url)
            doc_links.append((label, url))

    # 3) Fuera el ruido específico de la plantilla.
    for selector in _JUNK_SELECTORS:
        for el in soup.select(selector):
            el.decompose()

    # 4) Limpia cualquier "Copiar enlace" residual.
    for node in soup.find_all(string=re.compile(r"^\s*Copiar enlace\s*$")):
        if node.parent:
            node.parent.decompose()

    # 5) Texto + normalización de espacios.
    text = soup.get_text(separator="\n")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n\s*\n\s*\n+", "\n\n", text)

    # 6) Dedupe de líneas consecutivas idénticas.
    out: list[str] = []
    prev: str | None = None
    for line in text.splitlines():
        key = line.strip()
        if key and key == prev:
            continue
        out.append(line)
        prev = key
    body = "\n".join(out).strip()

    # 7) Apéndice con TODOS los documentos detectados (red de seguridad:
    #    aunque el LLM recorte algo inline, esta sección queda explícita).
    if doc_links:
        appendix = "\n\n## Documentos y enlaces oficiales\n" + "\n".join(
            f"- {label}: {url}" for label, url in doc_links
        )
        body = f"{body}{appendix}"

    return body

print("clean_ulima_html() definida.")

clean_ulima_html() definida.


## 4. Pulido de "chrome" de UI

Quita líneas que son solo decoración de los sliders (paginadores `1 / 3`,
botones `Siguiente`/`Previous`, toggles de idioma `Es`/`En`, etc.).

In [ ]:
_CHROME = re.compile(
    r"^(previous|siguiente|próximo|anterior|next|título|regresar|buscar|"
    r"es|en|\.|pasar al contenido principal|guía de accesibilidad.*|"
    r"menú.*|hamburger menu)$", re.I)
_PAGER = re.compile(r"^\d+\s*/\s*\d+$")   # "1 / 3", "1 / 5"


def polish_text(text: str) -> str:
    """Elimina líneas de chrome de UI (paginadores, 'Siguiente', toggles de idioma)."""
    keep = [
        ln for ln in text.splitlines()
        if not (_CHROME.match(ln.strip()) or _PAGER.match(ln.strip()))
    ]
    return re.sub(r"\n\s*\n\s*\n+", "\n\n", "\n".join(keep)).strip()


print("polish_text() definida.")

polish_text() definida.


## 5. Extractor principal

Descarga el HTML estático y aplica limpieza + pulido. Devuelve un `ScrapeResult`.

> Nota: la página es SSR, por eso `requests` basta. Si en el futuro adaptas esto
> a una página que sí requiera JavaScript, reemplaza el bloque `requests.get` por
> el HTML renderizado de Playwright (`await page.content()`) y mantén
> `clean_ulima_html` / `polish_text` igual.

In [ ]:
def scrape_ulima(url: str = TARGET_URL, polish: bool = True) -> ScrapeResult:
    """HTML estático (SSR) + limpieza quirúrgica (+ pulido opcional de chrome)."""
    method = "requests + limpieza" + (" + pulido" if polish else "")
    start = time.perf_counter()
    try:
        resp = requests.get(url, headers=HEADERS, timeout=REQUEST_TIMEOUT_S)
        resp.raise_for_status()
        resp.encoding = resp.apparent_encoding  # evita mojibake en acentos
        text = clean_ulima_html(resp.text, base_url=url)   # antes: clean_ulima_html(resp.text)
        if polish:
            text = polish_text(text)
        return ScrapeResult(method=method, text=text,
                            elapsed_s=time.perf_counter() - start)
    except Exception as exc:  # noqa: BLE001
        return ScrapeResult(method=method, error=f"{type(exc).__name__}: {exc}",
                            elapsed_s=time.perf_counter() - start)


# --- Ejecutar --------------------------------------------------------------
final = scrape_ulima()
print(f"{final.method}: {final.char_count:,} caracteres en {final.elapsed_s:.2f}s "
      f"({'OK' if final.ok else final.error})")
print("Repeticiones de 'Copiar enlace':", (final.text or "").count("Copiar enlace"))
print("-" * 70)
print((final.text or "")[:1200])

requests + limpieza + pulido: 6,723 caracteres en 1.28s (OK)
Repeticiones de 'Copiar enlace': 0
----------------------------------------------------------------------
Requisitos del programa Work and Travel USA | USE Peru

Inline HTML

Universal Student Exchange

Cambiar país: 

Regístrate.

Home

Quienes somos

Programas

PROMOCIONES

prensa

Galeria

Contáctanos

Work and Travel USA

Internship USA

Trainee USA

Home

Objetivos y beneficios

Requisitos

Inversión por opciones

Ferias de trabajo

Empleadores

Tipos de empleos

Cómo participar

Preguntas frecuentes

Requisitos 
para poder participar

Ser estudiante
 universitario o de instituto superior 
(1)

Tener disponibilidad para viajar en tus vacaciones de verano 
(2)

Tener 
entre 18 y 27
 años al 
30 de mayo del 2026.

Aprobar la Entrevista de evaluación en inglés, que evalúa tu nivel de inglés, sociabilidad y capacidad de adaptación al Programa 
(3)

Estar en buen estado de salud físico y mental para desempeñar un trabajo en l

In [ ]:
from __future__ import annotations

import asyncio
import textwrap
from typing import Final

import nest_asyncio
from google.colab import userdata
from openai import OpenAI

# Colab ya ejecuta un event loop propio; nest_asyncio permite anidar
# corutinas (necesario para Playwright/Crawl4AI dentro del notebook).
nest_asyncio.apply()

# --- Carga segura de la API Key desde los Secretos de Colab ---
# Crea el secreto en el panel 🔑 con el nombre EXACTO: OPENAI_ULIMA
try:
    OPENAI_API_KEY: str = userdata.get("OPENAI_ULIMA")
    assert OPENAI_API_KEY, "El secreto 'OPENAI_ULIMA' existe pero está vacío."
    print("✅ API Key 'OPENAI_ULIMA' cargada correctamente.")
except Exception as exc:  # noqa: BLE001
    raise RuntimeError(
        "No se pudo leer el secreto 'OPENAI_API_KEY'. Ábrelo en el panel 🔑 "
        "(Secrets) de Colab, créalo con ese nombre exacto y habilita el "
        "acceso para este notebook."
    ) from exc

# Cliente OpenAI reutilizable en todo el notebook
client: Final[OpenAI] = OpenAI(api_key=OPENAI_API_KEY)

✅ API Key 'OPENAI_ULIMA' cargada correctamente.


In [ ]:
import textwrap

# ============================================================
# CELDA 5 · Post-procesamiento: síntesis y estructuración RAG
# ============================================================

# Tope de seguridad de entrada. gpt-4o-mini soporta ~128k tokens de
# contexto; una página institucional rara vez se acerca a ese límite.
MAX_INPUT_CHARS: Final[int] = 60_000

SYSTEM_PROMPT: Final[str] = textwrap.dedent("""\
    Eres un procesador de datos especializado en preparar texto para una
    base de datos vectorial (sistema RAG) institucional de la Universidad
    de Lima.

    Tu única tarea es LIMPIAR y REESTRUCTURAR el texto crudo extraído de
    una página web. NO debes inventar, completar ni suponer información:
    trabaja exclusivamente con los datos provistos.

    Reglas estrictas:
    1. Elimina TODO el ruido de interfaz: menús de navegación, migas de
       pan (breadcrumbs), botones, banners de cookies, enlaces a redes
       sociales, textos legales repetidos, pies de página y llamados a la
       acción ("ver más", "inscríbete aquí", etc.). EXCEPCIÓN: conserva
       siempre las URLs que apunten a documentos descargables (.pdf, .doc,
       .docx, .xls, .xlsx, .ppt, .pptx) y a formularios/portales oficiales.
    2. Conserva ÍNTEGRAMENTE la información sustantiva: requisitos,
       convenios, universidades de destino, procesos y etapas del
       intercambio, fechas, plazos, contactos y documentos exigidos.
    3. Estructura el contenido jerárquicamente con Markdown semántico:
       - Un único H1 (#) con el tema general de la página.
       - H2 (##) y H3 (###) para secciones y subsecciones lógicas.
       - Listas con viñetas o numeradas para requisitos, pasos y enumeraciones.
       - Tablas Markdown SOLO si los datos son claramente tabulares.
    4. Redacta de forma concisa y AUTOCONTENIDA: cada sección debe poder
       entenderse por sí sola, sin depender del contexto visual de la web.
       Evita referencias ambiguas como "aquí", "esta página" o "más abajo".
    5. Mantén el idioma original (español) y la terminología institucional.
    6. Devuelve ÚNICAMENTE el Markdown final, sin comentarios ni
       explicaciones sobre tu proceso.
    7. NUNCA elimines ni acortes las URLs de documentos. Mantenlas literales
       junto al nombre del documento al que pertenecen. Si el texto incluye
       una sección "## Documentos y enlaces oficiales", consérvala íntegra.
""")


def process_text_for_rag(
    raw_text: str,
    client: OpenAI,
    *,
    model: str = "gpt-5.4-mini",
    temperature: float = 0.0,
) -> str:
    """Limpia y estructura el texto crudo para RAG usando OpenAI.

    Args:
        raw_text: Texto extraído por el scraper.
        client: Cliente OpenAI ya autenticado.
        model: Modelo de chat a usar (por defecto gpt-4o-mini).
        temperature: 0.0 para máxima fidelidad y reproducibilidad.

    Returns:
        Texto limpio y estructurado en Markdown semántico.

    Raises:
        ValueError: Si raw_text está vacío.
    """
    if not raw_text.strip():
        raise ValueError("raw_text está vacío; no hay nada que procesar.")

    payload = raw_text[:MAX_INPUT_CHARS]
    if len(raw_text) > MAX_INPUT_CHARS:
        print(
            f"⚠️ Texto truncado a {MAX_INPUT_CHARS:,} caracteres "
            f"(original: {len(raw_text):,})."
        )

    response = client.chat.completions.create(
        model=model,
        temperature=temperature,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {
                "role": "user",
                "content": (
                    "Procesa el siguiente texto crudo extraído de la web y "
                    "devuélvelo limpio y estructurado en Markdown:\n\n"
                    f"---\n{payload}\n---"
                ),
            },
        ],
    )

    content = response.choices[0].message.content
    return (content or "").strip()

In [ ]:
processed_text: str = process_text_for_rag(final.text, client)

print(f"✅ Procesamiento completado. Caracteres resultantes: {len(processed_text):,}")

✅ Procesamiento completado. Caracteres resultantes: 6,016


## 6. QA de cobertura

Verifica que el contenido de **cada programa y sub-tab** quedó capturado. Si todo
da ✅, la extracción está completa (no juzgar por los primeros caracteres).

## 7. Guardar el texto extraído

Guarda el resultado a disco para inspeccionarlo completo y para el siguiente paso
(chunking + embeddings).

In [ ]:
with open(OUTPUT_PATH, "w", encoding="utf-8") as fh:
    fh.write(processed_text or "")
print(f"💾 Guardado en '{OUTPUT_PATH}' ({final.char_count:,} caracteres).")
print("Ábrelo desde el panel de archivos de Colab (icono de carpeta a la izquierda).")

💾 Guardado en 'workuse - work-and-travel.txt' (6,723 caracteres).
Ábrelo desde el panel de archivos de Colab (icono de carpeta a la izquierda).


## Siguiente paso (fuera de este notebook)

Con `final.text` ya limpio, el flujo RAG continúa con:
1. **Chunking** semántico (p. ej. `RecursiveCharacterTextSplitter`, respetando
   encabezados).
2. **Embeddings** de cada chunk.
3. Carga en tu **base vectorial**.

Tip: como el texto conserva los títulos de sección, puedes usarlos como límites de
chunk o como metadatos para mejorar la recuperación.